# Análisis Exploratorio de Datos (EDA) - Bank Marketing

## Introducción

Este notebook realiza un Análisis Exploratorio de Datos (EDA) sobre el conjunto de datos de marketing bancario. El objetivo principal de este análisis es comprender mejor los datos, identificar patrones relevantes y obtener información que pueda ser útil para desarrollar un modelo de predicción del Retorno de la Inversión (ROI) en campañas de marketing.

## Estrategia Proxy de ROI

Para estimar el ROI, utilizaremos una estrategia proxy basada en los siguientes componentes:

1.  **Probabilidad de Suscripción:** Estimada por el modelo de clasificación que se desarrollará.
2.  **Ingreso Asumido por Suscripción:** Se asumirá un valor fijo de ingresos generado por cada nueva suscripción exitosa. Para este análisis, consideraremos un valor hipotético (por ejemplo, $100 MXN por suscripción).
3.  **Costo Asumido por Contacto:** Se asumirá un costo fijo asociado a cada contacto realizado durante la campaña de marketing (por ejemplo, $5 MXN por contacto).

El ROI proxy se calculará entonces como:

```
ROI_proxy = (Probabilidad_Suscripción * Ingreso_por_Suscripción_Asumido) - Costo_por_Contacto_Asumido
```

Este enfoque nos permitirá evaluar la rentabilidad esperada de contactar a un cliente potencial específico.

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración para visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 7) # Ajuste para gráficos más grandes
plt.rcParams['font.size'] = 10 # Ajuste de tamaño de fuente

## Carga y Exploración Inicial de los Datos

In [ ]:
# Descargar y descomprimir el conjunto de datos
# El archivo relevante es bank-full.csv
# !wget https://archive.ics.uci.edu/static/public/222/bank+marketing.zip -O bank_marketing.zip
# !unzip -o bank_marketing.zip bank-full.csv

# Cargar el archivo CSV en un DataFrame de pandas
# Asegúrate de que el archivo 'bank-full.csv' esté en el directorio correcto o proporciona la ruta completa.
try:
    df = pd.read_csv('bank-full.csv', sep=';')
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print("Error: El archivo 'bank-full.csv' no se encontró. Asegúrate de que esté en el directorio correcto.")
    print("Si no lo has hecho, descomenta y ejecuta las líneas de wget y unzip en la celda anterior para descargar los datos.")
    # df será None si el archivo no se encuentra, las celdas subsiguientes podrían fallar.
    df = None 

### Primeras filas del DataFrame

In [ ]:
# Mostrar las primeras filas del DataFrame
if df is not None:
    display(df.head())

### Información del DataFrame

In [ ]:
# Mostrar información general del DataFrame (tipos de datos, conteo de no nulos)
if df is not None:
    df.info()

### Estadísticas Descriptivas

In [ ]:
# Mostrar estadísticas descriptivas para todas las columnas (incluyendo categóricas)
if df is not None:
    display(df.describe(include='all'))

### Conteo de Valores Faltantes

In [ ]:
# Verificar la cantidad de valores nulos por columna
if df is not None:
    print("Conteo de valores nulos por columna:")
    display(df.isnull().sum())

### Dimensiones del DataFrame

In [ ]:
# Mostrar la forma del DataFrame (número de filas y columnas)
if df is not None:
    print(f"El DataFrame tiene {df.shape[0]} filas y {df.shape[1]} columnas.")

## 1. Análisis de la Variable Objetivo (`y`)

La variable objetivo `y` indica si el cliente se suscribió a un depósito a plazo ('yes') o no ('no').

In [ ]:
if df is not None:
    print("Conteo de valores para la variable objetivo 'y':")
    y_counts = df['y'].value_counts()
    print(y_counts)
    print(f"\nProporción de 'no': {y_counts['no'] / len(df):.2%}")
    print(f"Proporción de 'yes': {y_counts['yes'] / len(df):.2%}")
    
    plt.figure(figsize=(6,4))
    sns.countplot(x='y', data=df, palette=['#3498db', '#e74c3c'])
    plt.title('Distribución de la Variable Objetivo (y)', fontsize=15)
    plt.xlabel('Suscripción a Depósito a Plazo', fontsize=12)
    plt.ylabel('Cantidad de Clientes', fontsize=12)
    plt.show()

### Observaciones sobre la Variable Objetivo:
*   Existe un desbalance considerable en las clases. La gran mayoría de los clientes (`no`) no se suscribieron al depósito a plazo.
*   Aproximadamente el 88.30% de los clientes no se suscribieron, mientras que solo el 11.70% sí lo hizo.
*   Este desbalance es importante y deberá ser considerado durante el preprocesamiento y la modelización (e.g., usando técnicas de remuestreo o métricas de evaluación apropiadas).

## 2. Análisis Univariado

### 2.1 Variables Numéricas

In [ ]:
if df is not None:
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    # Excluir variables que podrían no ser continuas o que se analizan mejor como categóricas si es necesario
    # Por ahora, incluimos todas las numéricas identificadas por pandas
    print(f"Columnas numéricas identificadas: {numerical_cols}")
    
    for col in numerical_cols:
        plt.figure(figsize=(14, 5))
        
        # Histograma
        plt.subplot(1, 2, 1)
        sns.histplot(df[col], kde=True, color='skyblue')
        plt.title(f'Histograma de {col}', fontsize=14)
        plt.xlabel(col, fontsize=11)
        plt.ylabel('Frecuencia', fontsize=11)
        
        # Box plot
        plt.subplot(1, 2, 2)
        sns.boxplot(x=df[col], color='lightcoral')
        plt.title(f'Box Plot de {col}', fontsize=14)
        plt.xlabel(col, fontsize=11)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nObservaciones para '{col}':")
        if col == 'age':
            print("- La distribución de 'edad' está algo sesgada a la derecha. La mayoría de los clientes tienen entre 30 y 50 años. Hay algunos outliers en edades mayores.")
        elif col == 'balance':
            print("- 'balance' (saldo) está fuertemente sesgado a la derecha, con muchos clientes teniendo saldos bajos y algunos con saldos muy altos (outliers). Muchos clientes incluso tienen saldo negativo.")
        elif col == 'day':
            print("- 'day' (día del mes del contacto) parece distribuirse de manera relativamente uniforme, con picos en ciertos días, posiblemente relacionados con estrategias de campaña.")
        elif col == 'duration':
            print("- 'duration' (duración del último contacto) está muy sesgada a la derecha. La mayoría de los contactos son cortos. Hay outliers con duraciones muy largas. Nota: Esta variable tiene una alta influencia en el resultado (si la duración es 0, 'y' es 'no'), pero no se conoce hasta después de la llamada. Para un modelo predictivo realista, debería excluirse o manejarse con cuidado.")
        elif col == 'campaign':
            print("- 'campaign' (número de contactos durante esta campaña) está sesgada a la derecha. La mayoría de los clientes fueron contactados pocas veces. Hay outliers con un alto número de contactos.")
        elif col == 'pdays':
            print("- 'pdays' (días desde el último contacto de una campaña previa) tiene un valor predominante de -1 (cliente no contactado previamente). Para los contactados, la distribución está sesgada. Este valor -1 necesita un tratamiento especial.")
        elif col == 'previous':
            print("- 'previous' (número de contactos antes de esta campaña) está muy sesgada, con la mayoría en 0 (sin contactos previos). Similar a 'pdays', muchos no fueron contactados previamente.")
        print("-----\n")

### 2.2 Variables Categóricas

In [ ]:
if df is not None:
    categorical_cols = df.select_dtypes(include='object').columns.tolist()
    # La variable objetivo 'y' ya fue analizada, la removemos si está presente
    if 'y' in categorical_cols:
        categorical_cols.remove('y')
        
    print(f"Columnas categóricas identificadas (excluyendo 'y'): {categorical_cols}")
    
    for col in categorical_cols:
        plt.figure(figsize=(12, 6))
        order = df[col].value_counts().index # Ordenar por frecuencia
        sns.countplot(y=col, data=df, order=order, palette='viridis')
        plt.title(f'Distribución de {col}', fontsize=15)
        plt.xlabel('Cantidad de Clientes', fontsize=12)
        plt.ylabel(col, fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        print(f"\nObservaciones para '{col}':")
        if col == 'job':
            print("- Las profesiones más comunes son 'blue-collar', 'management', y 'technician'. Hay una categoría 'unknown'.")
        elif col == 'marital':
            print("- La mayoría de los clientes son 'married' (casados), seguidos por 'single' (solteros) y 'divorced' (divorciados). No hay valores 'unknown'.")
        elif col == 'education':
            print("- Los niveles educativos más frecuentes son 'secondary' y 'tertiary'. Existe una categoría 'unknown'.")
        elif col == 'default':
            print("- La gran mayoría de los clientes no tiene crédito en default ('no').")
        elif col == 'housing':
            print("- Un poco más de la mitad de los clientes tienen un préstamo hipotecario ('yes').")
        elif col == 'loan':
            print("- La mayoría de los clientes no tienen un préstamo personal ('no').")
        elif col == 'contact':
            print("- El tipo de contacto más común es 'cellular', seguido por 'unknown' (posiblemente contacto no registrado o no especificado) y 'telephone'. La categoría 'unknown' es significativa.")
        elif col == 'month':
            print("- Los meses con más contactos son 'may', 'jul', 'aug', 'jun'. La distribución no es uniforme, indicando picos de actividad de campaña.")
        elif col == 'poutcome':
            print("- 'poutcome' (resultado de la campaña previa) es mayoritariamente 'unknown', lo que es esperado ya que muchos clientes no fueron contactados previamente (ver 'pdays' y 'previous'). Otros resultados son 'failure', 'other', y 'success'.")
        print("-----\n")

## 3. Análisis Bivariado (Relación con la Variable Objetivo `y`)

### 3.1 Variables Numéricas vs. `y`

In [ ]:
if df is not None and numerical_cols is not None:
    print("Análisis Bivariado: Variables Numéricas vs. Variable Objetivo 'y'\n")
    for col in numerical_cols:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x='y', y=col, data=df, palette=['#3498db', '#e74c3c'])
        plt.title(f'{col} vs. Suscripción (y)', fontsize=15)
        plt.xlabel('Suscripción a Depósito a Plazo (y)', fontsize=12)
        plt.ylabel(col, fontsize=12)
        plt.tight_layout()
        plt.show()
        
        print(f"\nObservaciones para '{col}' vs. 'y':")
        if col == 'age':
            print("- Las distribuciones de edad para 'yes' y 'no' son similares, aunque la mediana de edad para los que se suscriben ('yes') parece ser ligeramente mayor. También hay suscriptores en rangos de edad más altos y más bajos.")
        elif col == 'balance':
            print("- Los clientes que se suscriben ('yes') tienden a tener un saldo ('balance') ligeramente más alto en promedio, aunque ambas categorías tienen muchos outliers y una gran dispersión.")
        elif col == 'day':
            print("- No parece haber una diferencia clara en el 'día' del contacto entre los que se suscriben y los que no, aunque ciertas campañas en días específicos podrían tener mejor o peor rendimiento.")
        elif col == 'duration':
            print("- 'duration' (duración del contacto) es un predictor muy fuerte. Llamadas más largas están fuertemente asociadas con una suscripción ('yes'). Esto es esperable, ya que si un cliente está interesado, la conversación será más larga. **Importante:** Como se mencionó antes, esta variable se conoce *después* del contacto, por lo que su uso directo en un modelo predictivo (para decidir *a quién* contactar) es problemático. Podría usarse para análisis post-campaña.")
        elif col == 'campaign':
            print("- Los clientes que se suscriben ('yes') tienden a haber sido contactados menos veces durante la campaña actual. Demasiados contactos podrían tener un efecto negativo.")
        elif col == 'pdays':
            print("- Para los clientes que fueron contactados previamente (pdays != -1), aquellos que se suscriben ('yes') tienden a tener un valor de 'pdays' menor (contacto previo más reciente). El grupo de 'pdays == -1' es muy grande y necesita un análisis separado o una transformación.")
        elif col == 'previous':
            print("- Los clientes que se suscriben ('yes') tienden a tener un mayor número de contactos previos ('previous'), aunque la mayoría de los clientes en ambas categorías tienen 0 contactos previos.")
        print("-----\n")

### 3.2 Variables Categóricas vs. `y`

In [ ]:
if df is not None and categorical_cols is not None:
    print("Análisis Bivariado: Variables Categóricas vs. Variable Objetivo 'y'\n")
    for col in categorical_cols:
        # Crear tabla de contingencia para proporciones
        contingency_table = pd.crosstab(df[col], df['y'], normalize='index') * 100
        contingency_table.rename(columns={'no':'Proporción No (%)', 'yes':'Proporción Yes (%)'}, inplace=True)
        
        ax = contingency_table.plot(kind='bar', stacked=False, figsize=(12,7), 
                                   color=['#3498db', '#e74c3c'])
        
        plt.title(f'Proporción de Suscripción (y) por {col}', fontsize=15)
        plt.xlabel(col, fontsize=12)
        plt.ylabel('Porcentaje de Clientes (%)', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Suscripción (y)', loc='upper right')
        plt.tight_layout()
        plt.show()
        
        print(f"\nObservaciones para '{col}' vs. 'y':")
        if col == 'job':
            print("- Las tasas de suscripción varían por profesión. 'student' y 'retired' parecen tener tasas de suscripción más altas. 'blue-collar' tiene una tasa relativamente baja.")
        elif col == 'marital':
            print("- Los clientes 'single' (solteros) tienen una tasa de suscripción ligeramente más alta que los 'married' (casados) o 'divorced' (divorciados). ")
        elif col == 'education':
            print("- Clientes con educación 'tertiary' (terciaria) muestran una tasa de suscripción más alta. La categoría 'unknown' tiene una tasa baja.")
        elif col == 'default':
            print("- No hay una diferencia apreciable, ya que casi nadie tiene default. Los pocos que sí tienen default, no se suscriben.")
        elif col == 'housing':
            print("- Los clientes sin préstamo hipotecario ('no') tienen una tasa de suscripción ligeramente más alta que aquellos con préstamo ('yes').")
        elif col == 'loan':
            print("- Los clientes sin préstamo personal ('no') tienen una tasa de suscripción notablemente más alta que aquellos con préstamo ('yes'). Esto sugiere que tener un préstamo personal disminuye la probabilidad de suscribirse.")
        elif col == 'contact':
            print("- El tipo de contacto 'cellular' tiene una tasa de suscripción más alta que 'telephone'. La categoría 'unknown' tiene la tasa más baja, lo que es esperable.")
        elif col == 'month':
            print("- Las tasas de suscripción varían significativamente por mes. Meses como 'mar', 'dec', 'sep', 'oct' muestran tasas de éxito mucho más altas, mientras que 'may' (el mes con más contactos) tiene una tasa baja. Esto sugiere estacionalidad o efectividad de campañas específicas.")
        elif col == 'poutcome':
            print("- El resultado de la campaña previa ('poutcome') es un fuerte indicador. 'success' en una campaña previa está asociado con una tasa de suscripción muy alta en la actual. 'failure' también muestra una tasa más alta que 'unknown', lo que podría indicar que cualquier contacto previo es mejor que ninguno, o que se seleccionaron para re-contacto.")
        print("-----\n")

## 4. Análisis de Correlación (Variables Numéricas)

In [ ]:
if df is not None and numerical_cols is not None:
    # Crear una copia para el análisis de correlación, por si se modifican los datos (ej. para 'pdays')
    df_corr = df.copy()
    
    # Convertir 'y' a numérico para incluirla en la correlación si se desea (opcional)
    # df_corr['y_numeric'] = df_corr['y'].apply(lambda x: 1 if x == 'yes' else 0)
    # numerical_cols_for_corr = numerical_cols + ['y_numeric']
    
    # Calcular la matriz de correlación solo con las columnas numéricas originales
    correlation_matrix = df_corr[numerical_cols].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
    plt.title('Mapa de Calor de Correlación de Variables Numéricas', fontsize=15)
    plt.show()
    
    print("\nObservaciones sobre la Correlación:")
    print("- La mayoría de las variables numéricas no muestran correlaciones fuertes entre sí, lo cual es bueno para evitar multicolinealidad en algunos modelos.")
    print("- Se observa una correlación moderada positiva entre 'pdays' y 'previous' cuando 'pdays' no es -1. Sin embargo, la forma en que 'pdays' está codificada (-1 para no contactados) afecta este cálculo directo. Para un análisis más preciso de su relación, se debería tratar el valor -1 de 'pdays' (por ejemplo, filtrándolo o convirtiéndolo a NaN o un número muy alto).")
    print("- 'duration' no está fuertemente correlacionada con otras variables predictoras, lo cual es interesante dado su fuerte impacto en la variable objetivo. (Recordar la advertencia sobre 'duration').")
    # print("- Si se incluyera 'y_numeric', se vería la correlación de 'duration' con 'y' como la más alta, seguida por 'previous' y 'pdays' (positivas). 'campaign' tendría una correlación negativa débil.")

## 5. Detección de Valores Atípicos (Outliers)

Basándonos en los box plots generados durante el análisis univariado de variables numéricas, podemos identificar varias características con una cantidad significativa de valores atípicos:

*   **`age` (edad):** Presenta algunos outliers en el extremo superior (edades muy avanzadas).
*   **`balance` (saldo):** Muestra una gran cantidad de outliers, tanto positivos (saldos muy altos) como negativos (saldos muy bajos/deudas). Esta variable está fuertemente sesgada.
*   **`duration` (duración del contacto):** Tiene muchos outliers en el extremo superior (llamadas muy largas).
*   **`campaign` (número de contactos):** Presenta outliers con un número elevado de contactos realizados a un mismo cliente.
*   **`pdays` (días desde el último contacto previo):** Cuando no es -1, muestra algunos outliers.
*   **`previous` (número de contactos previos):** También muestra outliers con un alto número de contactos previos.

Estos outliers pueden influir en el rendimiento de algunos modelos de machine learning. Será necesario considerar estrategias para manejarlos durante la etapa de preprocesamiento, como:
*   **Transformaciones:** Aplicar transformaciones logarítmicas o de potencia para reducir el sesgo y el impacto de los outliers (especialmente para `balance` y `duration`).
*   **Winsorización o Truncamiento:** Limitar los valores extremos a un cierto percentil.
*   **Eliminación:** Considerar la eliminación de outliers si son errores de datos o si son muy extremos y pocos (con precaución).
*   **Modelos robustos a outliers:** Utilizar modelos que sean inherentemente menos sensibles a los valores atípicos.

## 6. Resumen de Hallazgos Clave del EDA

Este Análisis Exploratorio de Datos ha revelado varias características importantes del conjunto de datos de marketing bancario:

**Hallazgos Principales y Tendencias:**
1.  **Desbalance de Clases:** La variable objetivo `y` está significativamente desbalanceada, con muchos más clientes que no se suscriben ('no') que los que sí lo hacen ('yes'). Esto requerirá atención en la modelización.
2.  **Importancia de `duration`:** La duración del último contacto (`duration`) es el predictor individual más fuerte de una suscripción. Sin embargo, su valor predictivo práctico es limitado ya que no se conoce antes de la llamada. Para un modelo predictivo de *quién contactar*, esta variable debe ser excluida o tratada con sumo cuidado (ej. para análisis de por qué una campaña fue exitosa *después* de hecha).
3.  **Impacto de Campañas Previas:** El resultado de campañas previas (`poutcome`) y el número de contactos previos (`previous`) tienen una influencia notable. Un `success` previo aumenta drásticamente la probabilidad de suscripción. Clientes contactados previamente (incluso con `failure` previo) parecen tener mayor propensión a suscribirse que los no contactados (`unknown` en `poutcome`).
4.  **Estacionalidad y Momento del Contacto:** El `month` del contacto es crucial, con tasas de éxito significativamente más altas en ciertos meses (ej. marzo, septiembre, octubre, diciembre), sugiriendo que el *timing* de las campañas es vital.
5.  **Características del Cliente:**
    *   Clientes `student` y `retired` (`job`) tienden a suscribirse más.
    *   Clientes `single` (`marital`) y con educación `tertiary` (`education`) muestran tasas de suscripción ligeramente mayores.
    *   Tener un préstamo personal (`loan`='yes') disminuye la probabilidad de suscripción. No tener un préstamo hipotecario (`housing`='no') la aumenta ligeramente.
6.  **Intensidad de la Campaña Actual:** Un número excesivo de contactos en la campaña actual (`campaign`) parece ser contraproducente; tasas de éxito menores se observan con más llamadas.
7.  **Valores 'Unknown':** Varias variables categóricas (`job`, `education`, `contact`, `poutcome`) contienen una proporción significativa de valores 'unknown'. Estos necesitarán ser manejados (imputación, tratados como una categoría separada, etc.). El `unknown` en `poutcome` es el más común y representa a clientes sin contacto previo.
8.  **Sesgo y Outliers:** Variables numéricas como `balance`, `duration`, y `campaign` están sesgadas y tienen muchos outliers, lo que podría requerir transformaciones (log, etc.) o técnicas de manejo de outliers.

**Características Prometedoras para la Predicción (excluyendo `duration` para predicción proactiva):**
*   `poutcome` (resultado de la campaña previa)
*   `month` (mes del contacto)
*   `previous` (número de contactos previos)
*   `contact` (tipo de comunicación de contacto)
*   `loan` (préstamo personal)
*   `housing` (préstamo hipotecario)
*   `job` (tipo de trabajo)
*   `education` (nivel educativo)
*   `campaign` (número de contactos en esta campaña, aunque con cuidado por posible no linealidad)

**Posibles Problemas de Calidad de Datos y Transformaciones Necesarias:**
*   **Manejo de 'unknown':** Decidir estrategia para valores desconocidos en variables categóricas.
*   **Manejo de `pdays = -1`:** Este valor indica "no contactado previamente" y domina la variable `pdays`. Necesita ser tratado como una categoría especial o transformado (e.g., a una variable binaria "contactado_previamente").
*   **Reducción de Sesgo:** Aplicar transformaciones (ej. logarítmica) a variables como `balance` para normalizar su distribución.
*   **Manejo de Outliers:** Aplicar técnicas para mitigar el efecto de outliers en variables como `balance`, `campaign`, `age`.
*   **Ingeniería de Características:** Podría ser útil crear nuevas características, por ejemplo, a partir de `pdays` (ej. si fue contactado o no), o combinando información.

Este EDA proporciona una base sólida para el preprocesamiento de datos y la selección de características para construir un modelo predictivo eficaz para la suscripción a depósitos a plazo y, posteriormente, estimar el ROI proxy.